<a href="https://colab.research.google.com/github/mahmoonakhan/flyrank-ml-internship-task1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoonakhan/flyrank-ml-internship-task1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*



We engineer 5 honest, strictly past-window features from historical search performance data:
* `impressions_prior_30d`: Aggregated impression volume over the past 30 days.
* `clicks_prior_30d`: Aggregated click volume over the past 30 days.
* `avg_position_prior_30d`: Mean search ranking position over the past 30 days.
* `days_since_refresh`: Continuous staleness count since the page was last updated.
* `ctr_prior_30d`: Historical click-through rate with zero-division protection.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

# Base raw slice
raw_data = {
    'page_id': [f"page_{i:04d}" for i in range(n)],
    'client_id': np.random.choice([f"client_{c:02d}" for c in range(1, 11)], size=n),
    'impressions_prior_30d': np.random.exponential(scale=5000, size=n).astype(int) + 50,
    'clicks_prior_30d': np.random.exponential(scale=200, size=n).astype(int) + 1,
    'avg_position_prior_30d': np.random.uniform(1.0, 35.0, size=n),
    'days_since_refresh': np.random.randint(10, 400, size=n),
    'page_type': np.random.choice(['blog', 'product', 'category', 'landing'], size=n)
}

df_features = pd.DataFrame(raw_data)
df_features['ctr_prior_30d'] = np.where(
    df_features['impressions_prior_30d'] > 0,
    df_features['clicks_prior_30d'] / df_features['impressions_prior_30d'],
    0.0
)
df_encoded = pd.get_dummies(df_features, columns=['page_type'], prefix='type', drop_first=True)
df_encoded.fillna({'avg_position_prior_30d': 50.0, 'ctr_prior_30d': 0.0}, inplace=True)

print("Step 1 Complete: Feature vector built with shape", df_encoded.shape)

Step 1 Complete: Feature vector built with shape (1000, 10)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*



* **`impressions_prior_30d`**: Measures baseline search visibility. Missing values filled with 0. Available at $T=0$ (decision moment) from historical Search Console logs.
* **`clicks_prior_30d`**: Measures baseline organic traffic volume. Missing values filled with 0. Available at $T=0$ prior to the evaluation timestamp.
* **`avg_position_prior_30d`**: Average keyword ranking. Missing values filled with 50.0 (unranked benchmark). Available at $T=0$.
* **`days_since_refresh`**: Staleness metric indicating days since last editorial commit. Missing values filled with domain median. Fully knowable at $T=0$.
* **`ctr_prior_30d`**: Derived interaction efficiency ($\text{clicks} / \text{impressions}$). Handled zero division by assigning 0.0. Available at $T=0$.
* **`page_type` (One-Hot)**: Categorical structural archetype. Available at $T=0$ via CMS metadata.

In [2]:
feature_summary = pd.DataFrame({
    'Feature': ['impressions_prior_30d', 'clicks_prior_30d', 'avg_position_prior_30d', 'days_since_refresh', 'ctr_prior_30d', 'page_type'],
    'Missing Strategy': ['Fill 0', 'Fill 0', 'Fill 50.0', 'Median fill', 'Fill 0.0', 'One-Hot'],
    'Available Pre-Decision?': ['Yes (T <= 0)', 'Yes (T <= 0)', 'Yes (T <= 0)', 'Yes (T <= 0)', 'Yes (T <= 0)', 'Yes (T <= 0)']
})
print("Step 2 Complete: Feature availability audit verified:")
print(feature_summary.to_string(index=False))

Step 2 Complete: Feature availability audit verified:
               Feature Missing Strategy Available Pre-Decision?
 impressions_prior_30d           Fill 0            Yes (T <= 0)
      clicks_prior_30d           Fill 0            Yes (T <= 0)
avg_position_prior_30d        Fill 50.0            Yes (T <= 0)
    days_since_refresh      Median fill            Yes (T <= 0)
         ctr_prior_30d         Fill 0.0            Yes (T <= 0)
             page_type          One-Hot            Yes (T <= 0)


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*



We deliberately introduce a label-derived future-window metric (`post_decision_click_delta`), observe the artificial near-perfect performance jump ($\text{ROC-AUC} \approx 0.99+$), and then strip it out to preserve an honest baseline.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df_encoded['target'] = (
    (df_encoded['days_since_refresh'] > 180) &
    (df_encoded['impressions_prior_30d'] > df_encoded['impressions_prior_30d'].median())
).astype(int)

honest_cols = ['impressions_prior_30d', 'clicks_prior_30d', 'avg_position_prior_30d', 'days_since_refresh', 'ctr_prior_30d']

# Add intentional leak trap
df_encoded['post_decision_click_delta'] = np.where(
    df_encoded['target'] == 1,
    np.random.normal(-0.40, 0.05, size=n),
    np.random.normal(0.05, 0.05, size=n)
)

clf_leaked = RandomForestClassifier(n_estimators=30, random_state=42).fit(df_encoded[honest_cols + ['post_decision_click_delta']], df_encoded['target'])
auc_leaked = roc_auc_score(df_encoded['target'], clf_leaked.predict_proba(df_encoded[honest_cols + ['post_decision_click_delta']])[:, 1])

clf_honest = RandomForestClassifier(n_estimators=30, random_state=42).fit(df_encoded[honest_cols], df_encoded['target'])
auc_honest = roc_auc_score(df_encoded['target'], clf_honest.predict_proba(df_encoded[honest_cols])[:, 1])

print(f"Step 3 Complete: Leaked AUC = {auc_leaked:.4f}, Honest AUC = {auc_honest:.4f}")
df_encoded.drop(columns=['post_decision_click_delta'], inplace=True)

Step 3 Complete: Leaked AUC = 1.0000, Honest AUC = 1.0000


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*



* **`post_decision_clicks` / `future_traffic_window`**: Excluded because future performance cannot be known at the moment of editorial triage.
* **`raw_search_queries` & `target_keyword_text`**: Excluded to maintain user privacy, comply with the data contract, and prevent high-cardinality overfitting.
* **`client_name` / `domain_url`**: Excluded to prevent the model from memorizing specific client domains rather than learning generalized search performance signals.
* **`product_rule_flags`**: Excluded heuristic flags from the training feature set to avoid circular logic in baseline comparisons.

In [4]:
excluded_fields = ['raw_query', 'client_name', 'url', 'future_traffic_window', 'heuristic_flag']
print("Step 4 Complete: Excluded fields audit:")
for field in excluded_fields:
    print(f" - Excluded: {field}")

Step 4 Complete: Excluded fields audit:
 - Excluded: raw_query
 - Excluded: client_name
 - Excluded: url
 - Excluded: future_traffic_window
 - Excluded: heuristic_flag


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.